In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

# 1) DATE

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text.replace("  "," ")
    return text

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = clean_text(sentence)

    ss = sid.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))

            # df.at[date, 'sentiment_complementary'] = 100 - 100*(ss['pos'] / (ss['pos'] + ss['neg']))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
        
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    print(date)
    df.at[date,"polarity"] = blob.sentiment.polarity
    df.at[date,"subjectivity"] = blob.sentiment.subjectivity
    if (blob.sentiment.polarity > 0 ):
        df.at[date,"polarityB"] = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        df.at[date,"polarityB"] = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        df.at[date,"polarityB"] = "Neutral"
    # except TypeError:
    #     #print("sentence is")
    #     #print(sentence)
    #     #print(df_vix.loc[date, 'articles'])
    #     print("ERROR:"+sentence)
    #     df.at[date, 'compound'] = 0
    #     df.at[date, 'neg'] = 0
    #     df.at[date, 'neu'] = 0
    #     df.at[date, 'pos'] = 0
    #     # df.at[date, 'sentiment'] = 0
    #     # df.at[date, 'sentiment_complementary'] = 0

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
df = df.join(df_articles)

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment.pkl")

# 2) HOUR

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDateHourDF.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
# df_articles.sort_index(inplace=True)

In [ ]:
aa = df_articles[["PressReleaseCount", "Company"]].loc[df_articles["PressReleaseCount"]>100]

In [ ]:
aa.

In [ ]:
df_articles["PressReleaseCount"].plot(figsize=(18, 6))

In [ ]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text.replace("  "," ")
    return text

rows_list = []
for date, row in df_articles.T.iteritems():

    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = clean_text(sentence)

    ss = sid.polarity_scores(sentence)

    Hour = row["Hour"]
    Compound = ss['compound']
    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    Sentiment=0
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            Sentiment = 100*(ss['pos'] / (ss['pos'] + ss['neg']))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    print(date)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Date":date, "Hour": Hour, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "sentiment":Sentiment, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "PressReleaseCount":row["PressReleaseCount"]}
    rows_list.append(new_row)

        # except TypeError:
        #     #print("sentence is")
        #     #print(sentence)
        #     #print(df_vix.loc[date, 'articles'])
        #     print("ERROR:"+sentence)
        #     df.at[date, 'compound'] = 0
        #     df.at[date, 'neg'] = 0
        #     df.at[date, 'neu'] = 0
        #     df.at[date, 'pos'] = 0
        #     # df.at[date, 'sentiment'] = 0
        #     # df.at[date, 'sentiment_complementary'] = 0

In [ ]:
df = pd.DataFrame(data=rows_list, columns=["Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","Articles","Title","Text", "Company", "PressReleaseCount"])

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
df["Articles"].tail(10)

In [ ]:
df

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourly.pkl")

# 3) TRADING HOURS 09-16

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_09_15.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text.replace("  "," ")
    return text

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = clean_text(sentence)

    ss = sid.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
            # df.at[date, 'sentiment_complementary'] = 100 - 100*(ss['pos'] / (ss['pos'] + ss['neg']))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
        
    # except TypeError:
    #     #print("sentence is")
    #     #print(sentence)
    #     #print(df_vix.loc[date, 'articles'])
    #     print("ERROR:"+sentence)
    #     df.at[date, 'compound'] = 0
    #     df.at[date, 'neg'] = 0
    #     df.at[date, 'neu'] = 0
    #     df.at[date, 'pos'] = 0
    #     # df.at[date, 'sentiment'] = 0
    #     # df.at[date, 'sentiment_complementary'] = 0

In [ ]:
df = df.join(df_articles)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_09_15.pkl")

# 4) Not Trading Hours 16-09

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_16_08.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text.replace("  "," ")
    return text

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = clean_text(sentence)

    ss = sid.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
            # df.at[date, 'sentiment_complementary'] = 100 - 100*(ss['pos'] / (ss['pos'] + ss['neg']))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
        
    # except TypeError:
    #     #print("sentence is")
    #     #print(sentence)
    #     #print(df_vix.loc[date, 'articles'])
    #     print("ERROR:"+sentence)
    #     df.at[date, 'compound'] = 0
    #     df.at[date, 'neg'] = 0
    #     df.at[date, 'neu'] = 0
    #     df.at[date, 'pos'] = 0
    #     # df.at[date, 'sentiment'] = 0
    #     # df.at[date, 'sentiment_complementary'] = 0

In [ ]:
df = df.join(df_articles)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_16_08.pkl")